# Data-loading tests

Exercises `qtnet.jax_models.representations` and the data-loading slice of
`scripts/atomic/train_multitask.py`:

* `row_to_complex` and `precompute_complexes` produce well-formed `Complex` objects.
* `create_cv_splits` yields 25 disjoint train/test partitions.
* `compute_per_atom_stats` + `apply_z_regularization` rescale per-atom features as
  expected.
* `prepare_padded_batches` assembles a `ComplexBatch` whose padded shapes line up
  with the true cell counts and target columns.

Synthetic data is built in `tests.tests` so the notebook runs without any external
files.


In [ ]:
import sys, os
if os.getcwd().endswith('tests'):
    sys.path.insert(0, os.getcwd())
else:
    sys.path.insert(0, os.path.join(os.getcwd(), 'tests'))

import pickle
import numpy as np
import pandas as pd

import tests as tt
from qtnet.data_utils import (
    create_cv_splits,
    compute_per_atom_stats,
    apply_z_regularization,
)
from qtnet.jax_models.representations import (
    row_to_complex,
    precompute_complexes,
    prepare_padded_batches,
)

ALL_ELEMENTS = tt.ALL_ELEMENTS
TARGET_COLUMNS = tt.TARGET_COLUMNS
ELEMENT_TO_IDX = {el: i for i, el in enumerate(ALL_ELEMENTS)}

runner = tt.TestRunner()


## 1. Synthetic dataframe

The factory mirrors the AIMEl atomic schema: `atom`, `position_x/y/z`, the ten
QTAIM target columns, and `Murcko_Scaffold`. We use 24 molecules across 6 distinct
scaffolds so `GroupKFold` produces non-trivial splits.


In [ ]:
df = tt.make_synthetic_dataframe(n_molecules=24, seed=0)
print(df.shape)
df.head(3)


## 2. `row_to_complex`

Build a single `Complex` with the production cutoff/max_neighbors and again with
`fully_connected=True`. The structural invariants check positions, edge atom
indices, and the absence of self-loops.


In [ ]:
def _row_to_complex_default():
    cx, _ = tt.make_minimal_complex(seed=0, n_atoms=9, fully_connected=False)
    tt.assert_complex_invariants(cx)

def _row_to_complex_fully_connected():
    cx, _ = tt.make_minimal_complex(seed=0, n_atoms=9, fully_connected=True)
    tt.assert_complex_invariants(cx)
    expected = 9 * 8 // 2
    assert cx.cochains[1].num_cells == expected, (
        f'expected {expected} edges, got {cx.cochains[1].num_cells}'
    )

runner.run('row_to_complex / cutoff=5.25', _row_to_complex_default)
runner.run('row_to_complex / fully_connected', _row_to_complex_fully_connected)


## 3. `precompute_complexes`

All df rows produce a `Complex`, the result is keyed by `df.index`, and a pickle
round-trip preserves the data.


In [ ]:
def _precompute_keys_match_index():
    complexes = precompute_complexes(
        df, element_to_idx=ELEMENT_TO_IDX,
        cutoff=5.25, max_neighbors=5, fully_connected=False,
        max_dim=2, verbose=False,
    )
    assert set(complexes.keys()) == set(df.index)
    for cx in complexes.values():
        tt.assert_complex_invariants(cx)

def _precompute_pickle_roundtrip():
    complexes = precompute_complexes(
        df, element_to_idx=ELEMENT_TO_IDX,
        cutoff=5.25, max_neighbors=5, fully_connected=False,
        max_dim=2, verbose=False,
    )
    blob = pickle.dumps(complexes)
    restored = pickle.loads(blob)
    assert restored.keys() == complexes.keys()
    cx_orig = complexes[df.index[0]]
    cx_back = restored[df.index[0]]
    np.testing.assert_array_equal(cx_orig.cochains[0].static['Z'], cx_back.cochains[0].static['Z'])
    np.testing.assert_allclose(cx_orig.cochains[0].static['pos'], cx_back.cochains[0].static['pos'])
    if cx_orig.cochains[1].num_cells > 0:
        np.testing.assert_allclose(cx_orig.cochains[1].static['G'], cx_back.cochains[1].static['G'])

runner.run('precompute_complexes / keys', _precompute_keys_match_index)
runner.run('precompute_complexes / pickle round-trip', _precompute_pickle_roundtrip)


## 4. `create_cv_splits` — 25 disjoint folds

Mirrors the `get_fold` helper used by `scripts/atomic/train_multitask.py:335`.


In [ ]:
def _cv_splits_disjoint():
    folds = list(create_cv_splits(df, group_col='Murcko_Scaffold'))
    assert len(folds) == 25, f'expected 25 folds, got {len(folds)}'
    tt.assert_cv_disjoint(folds, len(df))

runner.run('create_cv_splits / 25 disjoint folds', _cv_splits_disjoint)

# Pull fold 0 — the same logic the training script uses.
def get_fold(df, fold_index):
    for f in create_cv_splits(df, group_col='Murcko_Scaffold'):
        if f['fold'] == fold_index:
            return f['train_idx'], f['test_idx']
    raise ValueError(f'fold {fold_index} not found')

train_idx, val_idx = get_fold(df, 0)
train_df = df.iloc[train_idx].copy()
val_df = df.iloc[val_idx].copy()
print(f'fold 0: train={len(train_df)}, val={len(val_df)}')


## 5. Stats + z-regularization

Compute per-atom stats on the *training* slice, then regularize both train and
val. The synthetic dataset has no molecular-level columns, so we pass an empty
`mol_stats` and `mol_cols=[]`.

Post-regularization checks:
* per-element `N` and `LI` have ~zero mean and ~unit std on the training rows;
* per-element `Mu_*` and `Q_*` have ~unit RMS norm (each component is divided by
  the same scalar `Mu_rms` / `Q_rms`).


In [ ]:
def _stats_and_regularization():
    atomic_stats = compute_per_atom_stats(train_df)
    mol_stats = pd.DataFrame(index=['mean', 'std'])
    reg_train = apply_z_regularization(train_df.copy(), mol_stats, atomic_stats, mol_cols=[])
    tt.assert_post_regularization_stats(reg_train, atomic_stats)
    # Apply to val with the same stats — must not produce NaNs.
    reg_val = apply_z_regularization(val_df.copy(), mol_stats, atomic_stats, mol_cols=[])
    flat = pd.DataFrame(
        {col: np.concatenate([np.asarray(r) for r in reg_val[col]]) for col in TARGET_COLUMNS}
    )
    assert not flat.isna().any().any(), 'NaNs in regularized val rows'

runner.run('compute_per_atom_stats + apply_z_regularization', _stats_and_regularization)


## 6. `prepare_padded_batches`

Build padded batches from the regularized training slice and check that:

* shapes follow the `(real_cells_in_batch + 1)` convention with a trailing OOB row;
* `x_mask.sum()` equals the real number of cells in the batch;
* the per-atom target rows match the corresponding entries of the regularized
  training DataFrame in df-order.


In [ ]:
def _prepare_padded_batches():
    atomic_stats = compute_per_atom_stats(train_df)
    mol_stats = pd.DataFrame(index=['mean', 'std'])
    reg_train = apply_z_regularization(train_df.copy(), mol_stats, atomic_stats, mol_cols=[])
    complexes = precompute_complexes(
        df, element_to_idx=ELEMENT_TO_IDX,
        cutoff=5.25, max_neighbors=5, fully_connected=False,
        max_dim=2, verbose=False,
    )
    batches = prepare_padded_batches(
        complexes, reg_train, target_columns=TARGET_COLUMNS,
        batch_size=8, verbose=False, as_numpy=True,
    )
    assert len(batches) >= 1
    for b in batches:
        tt.assert_padded_batch_shapes(b, TARGET_COLUMNS)

    # Check target round-trip on the first batch.
    b0 = batches[0]
    node_batch = b0.cochain_batches[0]
    real = int(np.sum(node_batch.x_mask))
    expected_atoms = sum(
        complexes[idx].cochains[0].num_cells for idx in reg_train.index[:8]
    )
    assert real == expected_atoms, f'mask sum {real} != atoms {expected_atoms}'
    # First atom of first molecule
    first_idx = reg_train.index[0]
    expected_targets = np.array([reg_train.loc[first_idx, c][0] for c in TARGET_COLUMNS], dtype=np.float32)
    np.testing.assert_allclose(node_batch.y[0], expected_targets, atol=1e-6)

runner.run('prepare_padded_batches', _prepare_padded_batches)


## Summary


In [ ]:
ok = runner.report()
assert ok, 'data_loading.ipynb has failing tests'
